# Demo 2 — A2A

**What changes from Demo 1:** the agentic loop moves *inside* an agent we call over HTTP. The caller sends one sentence; the agent plans, fetches, writes a brief.

**Protocol stack — A2A wraps MCP:**

```
   you (this notebook)
        │  A2A protocol  (HTTP + JSON-RPC)
        │  one sentence:  "Analyze NVDA ..."
        ▼
   ┌─────────────────────────────────────────────────────┐
   │  analyst_agent.py  —  A2A server, port 9999         │
   │                                                     │
   │   ┌─ internal Claude loop (Anthropic SDK) ─┐        │
   │   │                                        │        │
   │   │      MCP protocol  (stdio JSON-RPC)    │        │
   │   │              │                         │        │
   │   │              ▼                         │        │
   │   │   stock_mcp_server.py  (subprocess)    │        │
   │   │              │  yfinance               │        │
   │   └────────────────────────────────────────┘        │
   └─────────────────────────────────────────────────────┘
        │
        ▼  written brief
   you (notebook prints it)
```

**Two protocols stacked.** The agent speaks A2A to the outside, MCP to its tools — that's exactly what `slides 13` and the side-by-side table mean by "complementary, not competing".

Helpers: [`a2a_helpers.py`](a2a_helpers.py) — pure HTTP / JSON-RPC plumbing for the client side.

The agent itself is **defined inline below** — no separate `analyst_agent.py`.

## First-time setup

Run this once in a terminal, from the directory where you want the repo:

```bash
git clone https://github.com/jackwu502/ivado-protocol.git
cd ivado-protocol

python3.12 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name ivado-lab --display-name "ivado-lab (3.12)"
```

Create your local `.env` file:

```bash
cp .env.example .env
```

Then edit `.env` and fill in one credential route:

```bash
# Option 1: Anthropic direct
ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_MODEL=claude-sonnet-4-6

# Option 2: OpenRouter-compatible Anthropic endpoint
# ANTHROPIC_BASE_URL=https://openrouter.ai/api
# ANTHROPIC_API_KEY=sk-or-v1-...
# ANTHROPIC_MODEL=anthropic/claude-sonnet-4.5
```

Do not commit `.env`; it is intentionally gitignored.

Start Jupyter from the repo root and select the `ivado-lab (3.12)` kernel:

```bash
python -m jupyter lab
```


Use the same `.env` from Demo 1. The A2A agent reads `ANTHROPIC_*` to call Claude internally.

In [ ]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")


In [ ]:
%pip install -q "a2a-sdk<1.0" uvicorn httpx anthropic mcp yfinance python-dotenv

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("ready")

## 1. The system prompt — the agent's role

This is plain English. The whole "intelligence" of the agent flows from this prompt + Claude's reasoning.

In [ ]:
SYSTEM_PROMPT = """You are an equity analyst.
When given a ticker or a high-level question, plan and fetch the data you
need (recent quote, price history, company info, news), then write a
concise brief: where the stock is, how it has moved, what the company does,
and what notable news may explain the move. Use the tools available."""

## 2. The agent's brain — `StockAnalystExecutor`

This class is the A2A `AgentExecutor` contract: `execute()` is called once per incoming task. Inside, we call `run_agent(...)` from `shared/agent_runner.py` — that's the Claude + MCP loop. **`mcp_servers=[STOCK_MCP_SERVER]` is the line where A2A nests MCP.**

We pass `trace=trace` so the agent's internal tool calls are captured and prepended to the response — useful for the demo.

In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message
from shared.agent_runner import run_agent


class StockAnalystExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        question = context.get_user_input() or "Analyze the market."
        trace: list[str] = []
        try:
            answer = await run_agent(
                question=question,
                mcp_servers=[STOCK_MCP_SERVER],     # ← A2A wraps MCP here
                system_prompt=SYSTEM_PROMPT,
                trace=trace,
            )
        except Exception as exc:
            answer = f"(analyst error: {exc})"
        if trace:
            full = (
                "── Internal trace (agent's own tool calls) ──\n"
                + "\n".join(trace)
                + "\n── End trace ──\n\n"
                + answer
            )
        else:
            full = answer
        await event_queue.enqueue_event(new_agent_text_message(full))

    async def cancel(self, context, event_queue):
        raise NotImplementedError("cancel not supported in this demo")

## 3. The agent card — self-description served at `/.well-known/agent-card.json`

Any A2A-aware client reads this card and knows the agent's name, skills, endpoint.

In [ ]:
from a2a.types import AgentCapabilities, AgentCard, AgentSkill


def build_agent_card() -> AgentCard:
    skill = AgentSkill(
        id="stock_brief",
        name="Stock brief",
        description=(
            "Given a ticker or a question about a stock, produce a brief "
            "covering recent price action, company profile, and notable news."
        ),
        tags=["finance", "equities", "research"],
        examples=[
            "Analyze NVDA.",
            "What's going on with TSLA this week?",
            "Brief me on MSFT — price, sector, recent news.",
        ],
    )
    return AgentCard(
        name="StockAnalystAgent",
        description="An equity analyst that produces written briefs from market data and news.",
        url="http://localhost:9999/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

print(build_agent_card().model_dump_json(indent=2, exclude_none=True))

## 4. Start the agent in-process

We run uvicorn as a background `asyncio` task **in this same kernel** — no subprocess, no external file. The HTTP server is live on `localhost:9999`.

In [ ]:
import asyncio
import uvicorn
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

handler = DefaultRequestHandler(
    agent_executor=StockAnalystExecutor(),
    task_store=InMemoryTaskStore(),
)
app = A2AStarletteApplication(agent_card=build_agent_card(), http_handler=handler)

config = uvicorn.Config(app.build(), host="0.0.0.0", port=9999, log_level="warning")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None  # we are not in the main thread
agent_task = asyncio.create_task(server.serve())
await asyncio.sleep(2)  # let it bind
print("agent running on http://localhost:9999")

## 5. Discover it over HTTP — fetch the agent card from outside

Note we `await` the helpers — the server runs in this kernel's event loop, so the client side has to be async too (sync httpx would deadlock the loop).

In [ ]:
from a2a_helpers import fetch_agent_card, print_agent_card, send_message, show_response

print_agent_card(await fetch_agent_card())

## 6. Delegate the task — one sentence in, one brief out

No tool list, no plan, no orchestration on our side. The agent figures it out internally.

In [ ]:
response = await send_message("Analyze NVDA — recent price action, what the company does, and any notable news.")
show_response(response)

### What just happened — the A2A jump from MCP

You sent **one sentence**. From your side: no tool list, no plan, no loop.

The reply has two parts:

- An **"Internal trace"** block at the top — captured by `trace=trace` in `StockAnalystExecutor.execute()` (cell 2 above).
- The actual brief below.

Look at the trace: the analyst called `get_quote`, then `get_history`, then `get_company_info`, then `get_news_headlines` — all on the stock MCP server, all decided by the analyst's own LLM. **None of that orchestration lives in this notebook's client-side code.**

That is the architectural shift A2A unlocks: the *callee* is a smart agent in its own right. In MCP your notebook's LLM was the orchestrator; here the LLM lives inside the executor we just defined, behind one HTTP call.

(In principle, an A2A agent can also call other A2A agents internally. The protocol does not prevent loops — depth limits, timeouts, and budgets are application concerns.)

## 7. Raw JSON-RPC envelope (optional)

Note `result.kind` and `result.status` — A2A models every call as a task with a lifecycle.

In [ ]:
show_response(response, raw=True)

## 8. Stop the agent

In [ ]:
server.should_exit = True
await asyncio.sleep(1)
agent_task.cancel()
print("agent stopped")